# Camada Gold

## Objetivo
A camada Gold tem como responsabilidade consolidar e agregar os dados tratados da camada Silver em tabelas analíticas orientadas ao consumo de negócio.

## O que este notebook entrega
As tabelas construídas neste notebook foram desenhadas para responder três grupos principais de perguntas:
- Visão 360 do cliente;
- Desempenho comercial e operacional dos produtos;
- Métricas analíticas prontas para consumo por frontend e Text-to-SQL.

## Decisões de modelagem
Para manter a solução simples, legível e segura para o case, adotamos as seguintes convenções:

- `dim_*` para dimensões analíticas;
- `dm_*` para data marts prontos para consumo;
- agregações sempre realizadas antes do join final, evitando fan-out e duplicação de métricas;
- preferência por tabelas com grão bem definido e uma linha por entidade principal.


In [0]:
%sql
USE CATALOG stack_overgol;
CREATE SCHEMA IF NOT EXISTS gold;

### 1. Grão das tabelas
Cada tabela Gold possui um grão explícito:
- `dim_cliente`: uma linha por cliente;
- `dim_produto`: uma linha por produto;
- `dm_cliente_360`: uma linha por cliente;
- `dm_produto_360`: uma linha por produto.

Definir o grão antes de escrever o SQL evita ambiguidades e reduz o risco de métricas infladas por joins indevidos.

### 2. Estratégia contra fan-out
Sempre que precisamos consolidar pedidos, tickets, avaliações e eventos, primeiro agregamos cada assunto separadamente e só depois fazemos o join final. Esse padrão foi adotado para evitar multiplicação de linhas em joins many-to-many, especialmente na visão 360 do cliente.

### 3. Princípio de consumo
A Gold foi modelada para ser facilmente consultada por:
- backend do CRM;
- dashboards analíticos;
- agente Text-to-SQL.

Por isso, priorizamos tabelas já consolidadas, com nomes claros e métricas prontas para uso.

### Dimensão de clientes
- **Objetivo** :A `dim_cliente` concentra os atributos cadastrais e demográficos do cliente, servindo como base para joins analíticos e filtros no CRM.

- **Grão**: Uma linha por cliente.

#### Principais transformações
Nesta tabela, mantemos a maior parte das informações vindas da Silver e acrescentamos apenas um atributo derivado de uso analítico:

- `idade_cliente`, calculada a partir da data de nascimento.

Não adicionamos métricas de compra ou suporte nesta dimensão, pois esses indicadores pertencem ao data mart de cliente 360.

In [0]:
%sql
CREATE OR REPLACE TABLE gold.dim_cliente AS
SELECT
    c.id_cliente,
    c.nome_cliente,
    c.sobrenome_cliente,
    CONCAT_WS(' ', c.nome_cliente, c.sobrenome_cliente) AS nome_completo_cliente,
    c.email_cliente,
    c.telefone_cliente,
    c.ramal_cliente,
    c.genero_cliente,
    c.data_nascimento_cliente,
    c.data_cadastro_cliente,
    c.endereco_cliente,
    c.cidade_cliente,
    c.estado_cliente,
    c.pais_cliente,
    c.origem_cliente,
    FLOOR(DATEDIFF(CURRENT_DATE(), c.data_nascimento_cliente) / 365.25) AS idade_cliente
FROM silver.clientes c;

num_affected_rows,num_inserted_rows


In [0]:
%sql
SELECT * FROM gold.dim_cliente

id_cliente,nome_cliente,sobrenome_cliente,nome_completo_cliente,email_cliente,telefone_cliente,ramal_cliente,genero_cliente,data_nascimento_cliente,data_cadastro_cliente,endereco_cliente,cidade_cliente,estado_cliente,pais_cliente,origem_cliente,idade_cliente
2e3d574b-57eb-4418-a778-9e84c7654689,Alan,Menezes,Alan Menezes,alan_menezes@1.com.br,(75) 9000-2424,3019,Masculino,1975-05-20,2021-10-30,639 Jennifer Creek,Salvador,BAHIA,Brasil,Web,50
4375cc64-a523-46b9-a87b-a34e5be6c014,Flavia,Dias,Flavia Dias,flavia_dias@hotmail.com,(86) 0319-5667,4475,Feminino,1980-12-18,2022-09-29,52798 CALEB MEADOW APT. 352,Camaçari,BAHIA,Brasil,Indicação,45
799232a9-ef75-4215-baeb-21e7a007d57c,Renan,Otero,Renan Otero,renan331@hotmail.com,(85) 2630-7452,null,Masculino,1998-03-14,2023-01-06,71826 Timothy Mills,Joaquim Nabuco,PERNAMBUCO,Brasil,Web,28
81d2afca-7f70-4906-acea-6a88b2609133,Hannah,Costa,Hannah Costa,hannah_costa@hotmail.com,(82) 0242-0982,544,Feminino,1967-09-09,2018-10-21,6952 Carol Hills Apt. 762,Goiânia,GOIÁS,Brasil,App,58
3711e720-6790-4915-9cda-a8759f7bf60d,Pedro,Quaresma,Pedro Quaresma,pedro_quaresma@gmail.com,(87) 8484-2580,49768,Masculino,1961-05-20,2023-12-19,923 Ryan Prairie,Guarulhos,SÃO PAULO,Brasil,Web,64
84a527bb-a443-47b6-89b4-78c9d8b1550f,Wesley,Serrano,Wesley Serrano,wesley.serrano195@1.com,(87) 1570-2726,null,Masculino,1999-08-12,2021-07-05,975 Roberts Grove Suite 836,São Paulo,SÃO PAULO,Brasil,Web,26
d9f437ed-e8d7-4707-93cc-c82bf1adc593,Julia,Queiroz,Julia Queiroz,julia681@1.com,(81) 4411-3387,4007,Feminino,1990-01-31,2023-07-29,1511 Williams Ferry Suite 056,Marcelândia,MATO GROSSO,Brasil,Web,36
bf2d0191-200c-47a1-b71c-e50204edc39e,Murilo,Evangelista,Murilo Evangelista,murilo266@hotmail.com,(19) 8558-5795,53928,Masculino,1979-04-19,2021-04-06,1277 Miranda Stream Suite 375,Sumaré,SÃO PAULO,Brasil,App,47
a7696c15-38ff-475d-8e3b-63c8c288f0cc,Zelinda,Amorim,Zelinda Amorim,zelinda541@hotmail.com,(92) 3737-9066,47062,Feminino,1980-01-12,2021-08-28,91423 Richard Branch Apt. 837,Itupiranga,PARÁ,Brasil,App,46
b06ff84e-3a74-4996-8323-e9ebc2273e48,Maria,Garcia,Maria Garcia,mgarcia538@yahoo.com,(87) 3421-5907,null,Feminino,1984-05-04,2022-05-30,9667 Butler Curve Suite 612,Mauá,SÃO PAULO,Brasil,Web,42


###Dimensão de produtos

- **Objetivo**: A `dim_produto` representa o catálogo analítico de produtos, reunindo atributos descritivos e classificações úteis para filtros e análises.
- **Grão**: Uma linha por produto.

#### Principais transformações
Além dos atributos vindos da Silver, criamos duas classificações derivadas:
- `faixa_preco_produto`, para segmentar produtos por faixa de preço;
- `status_estoque_produto`, para facilitar análises operacionais e exibição no CRM.

Essas classificações são simples, mas úteis para dashboards e consultas em linguagem natural pelo agente.

In [0]:
%sql
CREATE OR REPLACE TABLE gold.dim_produto AS
SELECT
    p.id_produto,
    p.nome_produto,
    p.categoria_produto,
    p.preco_produto,
    p.fornecedor_produto,
    p.peso_kg_produto,
    p.estoque_produto,
    p.produto_ativo,
    p.data_cadastro_produto,
    CASE
        WHEN p.preco_produto IS NULL THEN 'Nao Informado'
        WHEN p.preco_produto < 50 THEN 'Baixo'
        WHEN p.preco_produto < 200 THEN 'Medio'
        ELSE 'Alto'
    END AS faixa_preco_produto,
    CASE
        WHEN p.estoque_produto IS NULL THEN 'Nao Informado'
        WHEN p.estoque_produto = 0 THEN 'Sem Estoque'
        WHEN p.estoque_produto <= 10 THEN 'Estoque Baixo'
        ELSE 'Estoque Normal'
    END AS status_estoque_produto
FROM silver.catalogo_produtos p;

num_affected_rows,num_inserted_rows


###Data Mart de Cliente 360

**Objetivo**: A `dm_cliente_360` é a principal tabela analítica da camada Gold para consumo no CRM.

Ela consolida, em uma única linha por cliente, informações de cadastro, compras, suporte, avaliações e comportamento digital.

**Grão**: Uma linha por cliente.

#### Como foi feito
Para evitar distorções nas métricas, a tabela foi construída em duas etapas:
1. agregação individual de cada domínio de negócio (`pedidos`, `suporte_tickets`, `avaliacoes`, `clickstream`);
2. join final dessas agregações com a dimensão de cliente.

#### Por que essa estratégia é importante?
Se fizermos joins diretos entre pedidos, tickets e avaliações antes de agregar, um cliente com múltiplos registros em mais de uma tabela terá suas métricas multiplicadas indevidamente. Essa abordagem evita **fan-out** e **garante confiabilidade** para o CRM e para o agente Text-to-SQL.

#### Métricas principais
Entre os indicadores criados nesta tabela, destacam-se:

- total de pedidos;
- receita total do cliente;
- ticket médio;
- recência da última compra;
- total de tickets de suporte;
- NPS médio;
- total de sessões e eventos digitais;
- faixa de valor do cliente;
- indicador de cliente ativo nos últimos 90 dias.

In [0]:
%sql
CREATE OR REPLACE TABLE gold.dm_cliente_360 AS
WITH pedidos_agg AS (
    SELECT
        p.id_cliente,
        COUNT(DISTINCT p.id_pedido) AS total_pedidos,
        SUM(COALESCE(p.valor_pedido, 0)) AS receita_total_cliente,
        AVG(p.valor_pedido) AS ticket_medio_cliente,
        SUM(COALESCE(p.quantidade_produto, 0)) AS total_itens_comprados,
        MAX(p.data_pedido) AS data_ultima_compra,
        MIN(p.data_pedido) AS data_primeira_compra,
        COUNT(DISTINCT CASE WHEN p.status_pedido = 'Entregue' THEN p.id_pedido END) AS pedidos_entregues,
        COUNT(DISTINCT CASE WHEN p.status_pedido = 'Cancelado' THEN p.id_pedido END) AS pedidos_cancelados,
        COUNT(DISTINCT CASE WHEN p.status_pedido = 'Reembolsado' THEN p.id_pedido END) AS pedidos_reembolsados
    FROM silver.pedidos p
    GROUP BY p.id_cliente
),
tickets_agg AS (
    SELECT
        t.id_cliente,
        COUNT(DISTINCT t.ticket_id) AS total_tickets,
        COUNT(DISTINCT CASE WHEN t.status_ticket = 'Aberto' THEN t.ticket_id END) AS tickets_abertos,
        COUNT(DISTINCT CASE WHEN t.status_ticket = 'Fechado' THEN t.ticket_id END) AS tickets_fechados,
        AVG(t.tempo_resolucao_horas) AS tempo_medio_resolucao_horas,
        AVG(t.nota_avaliacao_problema) AS nota_media_atendimento,
        MAX(t.data_abertura) AS data_ultimo_ticket
    FROM silver.suporte_tickets t
    GROUP BY t.id_cliente
),
avaliacoes_agg AS (
    SELECT
        a.id_cliente,
        COUNT(DISTINCT a.id_avaliacao) AS total_avaliacoes,
        AVG(a.nota_produto) AS nota_media_produto,
        AVG(a.nota_nps) AS nps_medio_cliente,
        AVG(CASE WHEN a.recomenda_produto = TRUE THEN 1.0 ELSE 0.0 END) AS taxa_recomendacao_cliente,
        MAX(a.data_avaliacao) AS data_ultima_avaliacao
    FROM silver.avaliacoes a
    GROUP BY a.id_cliente
),
eventos_agg AS (
    SELECT
        e.id_cliente,
        COUNT(DISTINCT e.id_sessao) AS total_sessoes,
        COUNT(DISTINCT e.id_evento) AS total_eventos,
        AVG(e.tempo_pagina_seg) AS tempo_medio_pagina_seg,
        MAX(e.data_evento) AS data_ultimo_evento,
        COUNT(DISTINCT CASE WHEN e.tipo_evento = 'Compra' THEN e.id_evento END) AS eventos_compra,
        COUNT(DISTINCT CASE WHEN e.tipo_evento = 'Adicionar no Carrinho' THEN e.id_evento END) AS eventos_add_carrinho,
        COUNT(DISTINCT CASE WHEN e.tipo_evento = 'Vizualização de Página' THEN e.id_evento END) AS eventos_pageview
    FROM silver.clickstream e
    GROUP BY e.id_cliente
)
SELECT
    c.id_cliente,
    c.nome_cliente,
    c.sobrenome_cliente,
    c.nome_completo_cliente,
    c.email_cliente,
    c.telefone_cliente,
    c.genero_cliente,
    c.cidade_cliente,
    c.estado_cliente,
    c.pais_cliente,
    c.origem_cliente,
    c.data_cadastro_cliente,
    c.idade_cliente,

    COALESCE(p.total_pedidos, 0) AS total_pedidos,
    COALESCE(p.receita_total_cliente, 0) AS receita_total_cliente,
    p.ticket_medio_cliente,
    COALESCE(p.total_itens_comprados, 0) AS total_itens_comprados,
    p.data_primeira_compra,
    p.data_ultima_compra,
    DATEDIFF(CURRENT_DATE(), p.data_ultima_compra) AS recencia_dias,
    COALESCE(p.pedidos_entregues, 0) AS pedidos_entregues,
    COALESCE(p.pedidos_cancelados, 0) AS pedidos_cancelados,
    COALESCE(p.pedidos_reembolsados, 0) AS pedidos_reembolsados,

    COALESCE(t.total_tickets, 0) AS total_tickets,
    COALESCE(t.tickets_abertos, 0) AS tickets_abertos,
    COALESCE(t.tickets_fechados, 0) AS tickets_fechados,
    t.tempo_medio_resolucao_horas,
    t.nota_media_atendimento,
    t.data_ultimo_ticket,

    COALESCE(a.total_avaliacoes, 0) AS total_avaliacoes,
    a.nota_media_produto,
    a.nps_medio_cliente,
    a.taxa_recomendacao_cliente,
    a.data_ultima_avaliacao,

    COALESCE(e.total_sessoes, 0) AS total_sessoes,
    COALESCE(e.total_eventos, 0) AS total_eventos,
    e.tempo_medio_pagina_seg,
    e.data_ultimo_evento,
    COALESCE(e.eventos_compra, 0) AS eventos_compra,
    COALESCE(e.eventos_add_carrinho, 0) AS eventos_add_carrinho,
    COALESCE(e.eventos_pageview, 0) AS eventos_pageview,

    CASE
        WHEN COALESCE(p.receita_total_cliente, 0) >= 2000 THEN 'Alto Valor'
        WHEN COALESCE(p.receita_total_cliente, 0) >= 500 THEN 'Medio Valor'
        WHEN COALESCE(p.receita_total_cliente, 0) > 0 THEN 'Baixo Valor'
        ELSE 'Sem Compra'
    END AS faixa_valor_cliente,

    CASE
        WHEN p.data_ultima_compra >= DATE_SUB(CURRENT_DATE(), 90) THEN TRUE
        ELSE FALSE
    END AS cliente_ativo_90d

FROM gold.dim_cliente c
LEFT JOIN pedidos_agg p
    ON c.id_cliente = p.id_cliente
LEFT JOIN tickets_agg t
    ON c.id_cliente = t.id_cliente
LEFT JOIN avaliacoes_agg a
    ON c.id_cliente = a.id_cliente
LEFT JOIN eventos_agg e
    ON c.id_cliente = e.id_cliente;

num_affected_rows,num_inserted_rows


### Data Mart de Produto 360

**Objetivo**: A `dm_produto_360` foi criada para apoiar análises de desempenho comercial, satisfação e operação por produto.

Ela é especialmente útil para responder perguntas como:

- quais produtos mais vendem;
- quais produtos geram mais receita;
- quais produtos recebem mais tickets;
- quais produtos possuem pior avaliação;
- quais produtos têm maior engajamento digital.

**Grão**: Uma linha por produto.

#### Estratégia de construção
Assim como no mart de cliente, agregamos separadamente:
- vendas;
- avaliações;
- tickets vinculados a pedidos;
- eventos digitais do clickstream.

Somente após essas agregações fazemos o join final com a dimensão de produto.

### Métricas principais
Esta tabela reúne indicadores de venda, suporte, satisfação e navegação, o que torna seu uso muito adequado tanto para dashboards quanto para o agente de IA.

In [0]:
%sql
CREATE OR REPLACE TABLE gold.dm_produto_360 AS
WITH vendas_agg AS (
    SELECT
        p.id_produto,
        COUNT(DISTINCT p.id_pedido) AS total_pedidos,
        SUM(COALESCE(p.quantidade_produto, 0)) AS quantidade_vendida,
        SUM(COALESCE(p.valor_pedido, 0)) AS receita_total_produto,
        AVG(p.valor_pedido) AS ticket_medio_produto,
        MAX(p.data_pedido) AS data_ultima_venda,
        MIN(p.data_pedido) AS data_primeira_venda,
        COUNT(DISTINCT CASE WHEN p.status_pedido = 'Entregue' THEN p.id_pedido END) AS pedidos_entregues,
        COUNT(DISTINCT CASE WHEN p.status_pedido = 'Cancelado' THEN p.id_pedido END) AS pedidos_cancelados,
        COUNT(DISTINCT CASE WHEN p.status_pedido = 'Reembolsado' THEN p.id_pedido END) AS pedidos_reembolsados
    FROM silver.pedidos p
    GROUP BY p.id_produto
),
avaliacoes_agg AS (
    SELECT
        a.id_produto,
        COUNT(DISTINCT a.id_avaliacao) AS total_avaliacoes,
        AVG(a.nota_produto) AS nota_media_produto,
        AVG(a.nota_nps) AS nps_medio_produto,
        AVG(CASE WHEN a.recomenda_produto = TRUE THEN 1.0 ELSE 0.0 END) AS taxa_recomendacao_produto,
        MAX(a.data_avaliacao) AS data_ultima_avaliacao
    FROM silver.avaliacoes a
    GROUP BY a.id_produto
),
tickets_agg AS (
    SELECT
        p.id_produto,
        COUNT(DISTINCT t.ticket_id) AS total_tickets_produto,
        AVG(t.tempo_resolucao_horas) AS tempo_medio_resolucao_produto
    FROM silver.suporte_tickets t
    INNER JOIN silver.pedidos p
        ON t.id_pedido = p.id_pedido
    WHERE p.id_produto IS NOT NULL
    GROUP BY p.id_produto
),
eventos_agg AS (
    SELECT
        e.id_produto,
        COUNT(DISTINCT e.id_evento) AS total_eventos_produto,
        COUNT(DISTINCT e.id_sessao) AS total_sessoes_produto,
        COUNT(DISTINCT CASE WHEN e.tipo_evento = 'Vizualização de Página' THEN e.id_evento END) AS total_pageviews_produto,
        COUNT(DISTINCT CASE WHEN e.tipo_evento = 'Adicionar no Carrinho' THEN e.id_evento END) AS total_add_carrinho_produto,
        COUNT(DISTINCT CASE WHEN e.tipo_evento = 'Compra' THEN e.id_evento END) AS total_eventos_compra_produto,
        MAX(e.data_evento) AS data_ultimo_evento_produto
    FROM silver.clickstream e
    WHERE e.id_produto IS NOT NULL
    GROUP BY e.id_produto
)
SELECT
    dp.id_produto,
    dp.nome_produto,
    dp.categoria_produto,
    dp.preco_produto,
    dp.fornecedor_produto,
    dp.peso_kg_produto,
    dp.estoque_produto,
    dp.produto_ativo,
    dp.data_cadastro_produto,
    dp.faixa_preco_produto,
    dp.status_estoque_produto,

    COALESCE(v.total_pedidos, 0) AS total_pedidos,
    COALESCE(v.quantidade_vendida, 0) AS quantidade_vendida,
    COALESCE(v.receita_total_produto, 0) AS receita_total_produto,
    v.ticket_medio_produto,
    v.data_primeira_venda,
    v.data_ultima_venda,
    COALESCE(v.pedidos_entregues, 0) AS pedidos_entregues,
    COALESCE(v.pedidos_cancelados, 0) AS pedidos_cancelados,
    COALESCE(v.pedidos_reembolsados, 0) AS pedidos_reembolsados,

    COALESCE(a.total_avaliacoes, 0) AS total_avaliacoes,
    a.nota_media_produto,
    a.nps_medio_produto,
    a.taxa_recomendacao_produto,
    a.data_ultima_avaliacao,

    COALESCE(t.total_tickets_produto, 0) AS total_tickets_produto,
    t.tempo_medio_resolucao_produto,

    COALESCE(e.total_eventos_produto, 0) AS total_eventos_produto,
    COALESCE(e.total_sessoes_produto, 0) AS total_sessoes_produto,
    COALESCE(e.total_pageviews_produto, 0) AS total_pageviews_produto,
    COALESCE(e.total_add_carrinho_produto, 0) AS total_add_carrinho_produto,
    COALESCE(e.total_eventos_compra_produto, 0) AS total_eventos_compra_produto,
    e.data_ultimo_evento_produto,

    CASE
        WHEN COALESCE(v.receita_total_produto, 0) >= 10000 THEN 'Produto Destaque'
        WHEN COALESCE(v.receita_total_produto, 0) > 0 THEN 'Produto Ativo'
        ELSE 'Sem Venda'
    END AS status_comercial_produto,

    CASE
        WHEN COALESCE(t.total_tickets_produto, 0) >= 20 THEN TRUE
        ELSE FALSE
    END AS produto_com_alto_volume_suporte

FROM gold.dim_produto dp
LEFT JOIN vendas_agg v
    ON dp.id_produto = v.id_produto
LEFT JOIN avaliacoes_agg a
    ON dp.id_produto = a.id_produto
LEFT JOIN tickets_agg t
    ON dp.id_produto = t.id_produto
LEFT JOIN eventos_agg e
    ON dp.id_produto = e.id_produto;

num_affected_rows,num_inserted_rows


### Validação rápida das tabelas Gold

Executamos, então, consultas de sanidade para validar:

- se as tabelas foram criadas corretamente;
- se o grão esperado foi respeitado;
- se há volume plausível de registros;
- se os joins finais não duplicaram clientes ou produtos.

Essas verificações não substituem testes formais de qualidade, mas ajudam a detectar erros estruturais logo no desenvolvimento.

In [0]:
%sql
SELECT COUNT(*) AS total_linhas, COUNT(DISTINCT id_cliente) AS clientes_unicos
FROM gold.dm_cliente_360;

total_linhas,clientes_unicos
58322,58322


In [0]:
%sql
SELECT COUNT(*) AS total_linhas, COUNT(DISTINCT id_produto) AS produtos_unicos
FROM gold.dm_produto_360;

total_linhas,produtos_unicos
517,517


In [0]:
%sql
SELECT *
FROM gold.dm_cliente_360
LIMIT 10;

id_cliente,nome_cliente,sobrenome_cliente,nome_completo_cliente,email_cliente,telefone_cliente,genero_cliente,cidade_cliente,estado_cliente,pais_cliente,origem_cliente,data_cadastro_cliente,idade_cliente,total_pedidos,receita_total_cliente,ticket_medio_cliente,total_itens_comprados,data_primeira_compra,data_ultima_compra,recencia_dias,pedidos_entregues,pedidos_cancelados,pedidos_reembolsados,total_tickets,tickets_abertos,tickets_fechados,tempo_medio_resolucao_horas,nota_media_atendimento,data_ultimo_ticket,total_avaliacoes,nota_media_produto,nps_medio_cliente,taxa_recomendacao_cliente,data_ultima_avaliacao,total_sessoes,total_eventos,tempo_medio_pagina_seg,data_ultimo_evento,eventos_compra,eventos_add_carrinho,eventos_pageview,faixa_valor_cliente,cliente_ativo_90d
08989252-01bd-4d64-bb0f-f94aa4ce3815,Denise,Martins,Denise Martins,denise316@yahoo.com,(91) 0106-6859,Feminino,Arapiraca,ALAGOAS,Brasil,Web,2021-11-11,42,6,9342.27,1557.045,12,2023-11-18,2026-03-26,41,0,0,1,1,0,0,146.0,4.0,2026-03-28T20:11:00.000Z,2,4.5,8.5,1.00000,2026-04-19,3,3,186.66666666666666,2025-07-04T15:24:35.000Z,0,0,1,Alto Valor,true
21a1b1c8-7ebb-4546-a2c0-08d535d5a135,Amelia,Siqueira,Amelia Siqueira,amelia.siqueira130@hotmail.com,(89) 6307-8067,Feminino,São Paulo,PORTO FERREIRA,Brasil,Web,2022-12-28,49,6,2327.99,387.9983333333333,18,2023-05-15,2026-03-10,57,0,0,0,3,0,0,133.5,3.0,2026-03-31T23:11:00.000Z,2,5.0,9.0,1.00000,2026-03-27,7,7,196.85714285714286,2026-01-08T16:47:08.000Z,0,1,2,Alto Valor,true
903663ce-31dc-451d-afd3-71e49c44210d,Ulisses,Ortiz,Ulisses Ortiz,ulisses736@hotmail.com,(79) 2755-0047,Masculino,Jacareí,SÃO PAULO,Brasil,Web,2022-04-24,45,4,10873.119999999999,2718.2799999999997,3,2024-01-12,2025-12-22,135,0,0,0,0,0,0,null,null,null,3,3.0,6.333333333333333,0.66667,2026-01-03,4,4,277.0,2025-09-24T22:11:34.000Z,0,0,4,Alto Valor,false
93a7b32d-28fb-4781-a825-d20b195cab94,Fernanda,Miranda,Fernanda Miranda,fernanda_miranda@yahoo.com,(16) 5892-8192,Feminino,Sete Lagoas,MINAS GERAIS,Brasil,Web,2019-05-18,31,2,6126.12,3063.06,7,2023-08-08,2024-03-17,780,0,0,0,0,0,0,null,null,null,2,2.5,9.0,1.00000,2024-05-13,7,7,291.57142857142856,2026-01-10T00:02:43.000Z,0,0,7,Alto Valor,false
b1b7c52d-e016-42d8-b5f7-7ec25defd534,Iris,Jesus,Iris Jesus,iris.jesus@hotmail.com,(93) 8172-1509,Feminino,Ilhéus,BAHIA,Brasil,Web,2023-04-24,77,3,751.27,751.27,11,2023-05-03,2025-04-14,387,0,0,0,0,0,0,null,null,null,2,3.5,5.5,0.50000,2025-02-05,8,8,366.75,2026-01-20T12:15:29.000Z,0,0,4,Medio Valor,false
cc5963b4-f9b3-4520-8df0-8796bfec5190,Xavier,Moreira,Xavier Moreira,xavier351@gmail.com,(84) 2908-2169,Masculino,Cajamar,SÃO PAULO,Brasil,App,2023-01-03,53,7,7344.91,1049.2728571428572,12,2023-07-28,2026-05-07,-1,0,0,0,0,0,0,null,null,null,4,4.0,8.75,1.00000,2024-12-06,6,6,367.2,2025-09-10T20:17:21.000Z,0,0,2,Alto Valor,true
84e96d88-cd41-4335-ac1d-c0aa776db948,Barbara,Ribeiro,Barbara Ribeiro,bribeiro118@yahoo.com,(81) 6164-4621,Feminino,Uberaba,MINAS GERAIS,Brasil,Indicação,2023-05-31,36,4,10931.699999999999,2732.9249999999997,14,2024-09-03,2025-09-23,225,0,0,2,0,0,0,null,null,null,2,2.5,5.5,0.50000,2025-11-21,8,8,169.42857142857142,2025-06-20T09:36:29.000Z,0,1,4,Alto Valor,false
c4d2ef01-2b24-4866-b7ba-563c230bd23f,Quirina,Xisto,Quirina Xisto,quirina_xisto@1.com.br,(95) 1238-4765,Feminino,Braço do Norte,SANTA CATARINA,Brasil,Indicação,2021-04-12,27,10,12257.37,1225.737,28,2023-01-05,2025-06-20,320,0,0,0,2,0,0,88.0,4.0,2024-01-31T12:17:00.000Z,7,3.857142857142857,7.714285714285714,0.71429,2025-06-11,12,12,192.4,2025-12-16T19:46:35.000Z,1,2,5,Alto Valor,false
eeb12e34-b5a3-40c6-be93-a17dd3406240,Leandro,Santos,Leandro Santos,leandro.santos@yahoo.com,(94) 1503-6986,Masculino,São Pedro da Aldeia,RIO DE JANEIRO,Brasil,App,2022-12-03,42,7,11818.91,1688.4157142857143,22,2024-04-13,2025-12-24,133,0,0,0,1,0,0,187.0,5.0,2025-12-04T03:15:00.000Z,4,4.0,8.25,0.75000,2025-06-07,6,6,256.6666666666667,2025-12-16T03:19:31.000Z,0,2,0,Alto Valor,false
8e4b2433-0d1d-49cc-a5bd-5f8f2353d80e,Wende

In [0]:
%sql
SELECT *
FROM gold.dm_produto_360
LIMIT 10;

id_produto,nome_produto,categoria_produto,preco_produto,fornecedor_produto,peso_kg_produto,estoque_produto,produto_ativo,data_cadastro_produto,faixa_preco_produto,status_estoque_produto,total_pedidos,quantidade_vendida,receita_total_produto,ticket_medio_produto,data_primeira_venda,data_ultima_venda,pedidos_entregues,pedidos_cancelados,pedidos_reembolsados,total_avaliacoes,nota_media_produto,nps_medio_produto,taxa_recomendacao_produto,data_ultima_avaliacao,total_tickets_produto,tempo_medio_resolucao_produto,total_eventos_produto,total_sessoes_produto,total_pageviews_produto,total_add_carrinho_produto,total_eventos_compra_produto,data_ultimo_evento_produto,status_comercial_produto,produto_com_alto_volume_suporte
PROD-0340,Capa de Chuva para Carro,Automotivo,248.0,Automotivo Import,21.9,479,false,2018-09-23,Alto,Estoque Normal,295,783,152190.05999999988,580.878091603053,2023-01-15,2026-05-18,0,0,4,138,3.717391304347826,7.4855072463768115,0.78261,2026-05-28,22,131.47368421052633,526,524,331,98,7,2026-03-26T06:59:39.000Z,Produto Destaque,true
PROD-0300,Kit Barba Completo,Beleza,158.0,Beauty Import Brasil,0.83,0,false,2023-03-04,Medio,Sem Estoque,306,781,96861.82999999993,342.2679505300351,2023-01-04,2026-05-31,0,0,7,143,3.6363636363636362,7.244755244755245,0.76224,2026-07-21,21,110.7,506,506,329,93,5,2026-03-25T16:13:26.000Z,Produto Destaque,true
PROD-0371,Pastilha de Freio Kit,Outros,208.0,Peças & Cia Distribuidora,21.7,164,true,2020-02-03,Alto,Estoque Normal,269,745,119722.70000000001,482.7528225806452,2023-01-06,2026-05-29,0,0,7,123,3.5772357723577235,6.983739837398374,0.73171,2026-06-18,23,147.94736842105263,518,517,357,72,8,2026-03-28T17:43:29.000Z,Produto Destaque,true
PROD-0166,Umidificador de Ar,Casa,null,Decor Import,5.56,33,true,2019-12-12,Nao Informado,Estoque Normal,562,1539,114292.14000000007,224.10223529411778,2023-01-02,2026-05-27,0,0,18,280,3.6607142857142856,7.260714285714286,0.74286,2026-06-16,59,112.98245614035088,550,549,343,98,9,2026-03-27T19:30:25.000Z,Produto Destaque,true
PROD-0469,Comoda 3 Gavetas,Móveis,null,MoveisPlus Distribuidora,23.07,299,true,2026-03-05,Nao Informado,Estoque Normal,621,1709,135261.53999999998,234.42207972270361,2023-01-07,2026-05-12,0,0,17,305,3.685245901639344,7.324590163934427,0.75082,2026-06-06,44,115.5813953488372,532,528,352,82,12,2026-03-28T23:53:58.000Z,Produto Destaque,true
PROD-0147,Cortina Blackout 2 Peças,Casa,178.0,Decor Import,10.92,0,true,2019-01-30,Medio,Sem Estoque,299,838,113095.82999999994,415.7934926470586,2023-01-09,2026-05-29,0,0,12,161,3.6459627329192545,7.149068322981367,0.73292,2026-06-04,28,122.89285714285714,518,517,333,85,5,2026-03-26T08:38:32.000Z,Produto Destaque,true
PROD-0194,Bola de Futebol Profissional,Esportes,129.0,SportsBrasil Atacado,7.81,138,true,2018-09-10,Medio,Estoque Normal,291,798,76027.07000000004,292.41180769230783,2023-01-03,2026-05-29,0,0,9,159,3.6666666666666665,7.088050314465409,0.71698,2026-06-30,25,103.79166666666667,586,582,378,111,14,2026-03-25T14:25:09.000Z,Produto Destaque,true
PROD-0467,Criado Mudo,Móveis,464.0,MoveisPlus Distribuidora,17.6,481,true,2024-11-09,Alto,Estoque Normal,132,365,127123.41000000003,1068.2639495798321,2023-01-02,2026-05-01,0,0,5,68,3.426470588235294,6.661764705882353,0.64706,2026-05-31,6,138.16666666666666,548,544,346,96,7,2026-03-27T21:22:28.000Z,Produto Destaque,false
PROD-0510,Suporte para Moto,Automotivo,107.0,Automotivo Import,2.61,241,true,2022-02-14,Medio,Estoque Normal,299,804,66268.17000000003,244.5319926199263,2023-01-06,2026-05-28,0,0,6,155,3.8193548387096774,7.670967741935484,0.81935,2026-07-21,19,123.0,0,0,0,0,0,null,Produto Destaque,false
PROD-0256,Chapinha Profissional,Beleza,303.0,Beleza Total Atacado,1.29,0,true,2021-12-18,Alto,Sem Estoque,126,352,85242.23999999996,722.3918644067793,2023-01-17,2026-05-29,0,0,4,58,3.706896551724138,7.103448275862069,0.74138,2026-07-01,13,113.15384615384616,477,475,310,89,4,2026-03-24T07:11:00.000Z,Produto Destaque,false


#### Otimização física

Como as tabelas Gold serão consultadas pelo CRM e pelo agente de IA, aplicamos otimizações físicas apenas nas tabelas de maior valor analítico.
O objetivo aqui não é complexidade excessiva, mas melhorar leitura e organização dos arquivos Delta para os principais padrões de acesso.

In [0]:
%sql
OPTIMIZE gold.dim_cliente
ZORDER BY (id_cliente);

OPTIMIZE gold.dim_produto
ZORDER BY (id_produto);

OPTIMIZE gold.dm_cliente_360
ZORDER BY (id_cliente, estado_cliente);

OPTIMIZE gold.dm_produto_360
ZORDER BY (id_produto, categoria_produto);

path,metrics
,"List(0, 0, List(null, null, 0.0, 0, 0), List(null, null, 0.0, 0, 0), 0, List(minCubeSize(107374182400), List(0, 0), List(1, 53938), 0, List(0, 0), 0, null), null, 0, 0, 1, 1, false, 0, 0, 1778091631261, 1778091631770, 8, 0, null, List(0, 0), null, 35, 32, 0, 0, null, null)"


###Data Mart de vendas por período

#### Objetivo
A `dm_vendas_periodo` foi criada para atender principalmente as necessidades do dashboard comercial e das análises agregadas feitas pelo CRM.

Ela resume as vendas em um grão mais analítico do que a visão transacional de pedidos, permitindo responder rapidamente perguntas como:

- como está a receita por dia;
- qual categoria mais vendeu em determinado período;
- qual estado concentrou mais pedidos;
- como evoluíram ticket médio e volume de vendas.

#### Grão
Uma linha por combinação de:

- data do pedido;
- estado do cliente;
- categoria do produto;
- status do pedido;
- método de pagamento.

#### Benefícios
Esse desenho facilita filtros por período, categoria, status e região, que são requisitos explícitos do case para o CRM.

In [0]:
%sql
CREATE OR REPLACE TABLE gold.dm_vendas_periodo AS
SELECT
    p.data_pedido,
    YEAR(p.data_pedido) AS ano_pedido,
    MONTH(p.data_pedido) AS mes_pedido,
    DATE_TRUNC('month', p.data_pedido) AS mes_referencia,
    c.estado_cliente,
    c.cidade_cliente,
    pr.categoria_produto,
    p.status_pedido,
    p.metodo_pagamento,

    COUNT(DISTINCT p.id_pedido) AS total_pedidos,
    COUNT(DISTINCT p.id_cliente) AS total_clientes,
    COUNT(DISTINCT p.id_produto) AS total_produtos,
    SUM(COALESCE(p.quantidade_produto, 0)) AS total_itens,
    SUM(COALESCE(p.valor_pedido, 0)) AS receita_total,
    AVG(p.valor_pedido) AS ticket_medio,
    SUM(CASE WHEN p.status_pedido = 'Entregue' THEN COALESCE(p.valor_pedido, 0) ELSE 0 END) AS receita_entregue,
    SUM(CASE WHEN p.status_pedido = 'Cancelado' THEN COALESCE(p.valor_pedido, 0) ELSE 0 END) AS receita_cancelada,
    SUM(CASE WHEN p.status_pedido = 'Reembolsado' THEN COALESCE(p.valor_pedido, 0) ELSE 0 END) AS receita_reembolsada

FROM silver.pedidos p
LEFT JOIN gold.dim_cliente c
    ON p.id_cliente = c.id_cliente
LEFT JOIN gold.dim_produto pr
    ON p.id_produto = pr.id_produto
WHERE p.data_pedido IS NOT NULL
GROUP BY
    p.data_pedido,
    YEAR(p.data_pedido),
    MONTH(p.data_pedido),
    DATE_TRUNC('month', p.data_pedido),
    c.estado_cliente,
    c.cidade_cliente,
    pr.categoria_produto,
    p.status_pedido,
    p.metodo_pagamento;

num_affected_rows,num_inserted_rows


###Exportação das tabelas Gold
Após a construção das tabelas analíticas da camada Gold, exportamos os resultados em CSV para servir como insumo do banco local utilizado pelo backend do CRM e pelo agente de IA.

#### Como foi feito
Como o volume das tabelas Gold é significativamente menor do que o das tabelas de origem, utilizamos `coalesce(1)` para gerar um único arquivo CSV por tabela, simplificando a carga no banco local.

#### Tabelas exportadas
Nesta etapa, exportamos:

- `dim_cliente`
- `dim_produto`
- `dm_cliente_360`
- `dm_produto_360`
- `dm_vendas_periodo`

Criamos um volume dedicado chamado `gold_exports`, separado da zona de landing, para manter a organização do projeto.
Cada tabela será exportada para um subdiretório próprio dentro do volume.

In [0]:
%sql
USE CATALOG stack_overgol;

CREATE SCHEMA IF NOT EXISTS default;

CREATE VOLUME IF NOT EXISTS stack_overgol.default.gold_exports;

In [0]:
%sql
SHOW VOLUMES IN stack_overgol.default;
-- SHOW TABLES IN gold

database,volume_name
default,gold_exports
default,landing


In [0]:
from pyspark.sql import DataFrame

EXPORT_BASE_PATH = "/Volumes/stack_overgol/default/gold_exports"

tables_to_export = [
    "gold.dim_cliente",
    "gold.dim_produto",
    "gold.dm_cliente_360",
    "gold.dm_produto_360",
    "gold.dm_vendas_periodo"
]

for table_name in tables_to_export:
    df = spark.table(f"stack_overgol.{table_name}")
    export_name = table_name.split(".")[-1]
    output_path = f"{EXPORT_BASE_PATH}/{export_name}"

    (
        df.coalesce(1)
          .write
          .mode("overwrite")
          .option("header", True)
          .csv(output_path)
    )

    print(f"Tabela exportada com sucesso: {table_name} -> {output_path}")

Tabela exportada com sucesso: gold.dim_cliente -> /Volumes/stack_overgol/default/gold_exports/dim_cliente
Tabela exportada com sucesso: gold.dim_produto -> /Volumes/stack_overgol/default/gold_exports/dim_produto
Tabela exportada com sucesso: gold.dm_cliente_360 -> /Volumes/stack_overgol/default/gold_exports/dm_cliente_360
Tabela exportada com sucesso: gold.dm_produto_360 -> /Volumes/stack_overgol/default/gold_exports/dm_produto_360
Tabela exportada com sucesso: gold.dm_vendas_periodo -> /Volumes/stack_overgol/default/gold_exports/dm_vendas_periodo


###Validação da exportação
Após a escrita dos arquivos CSV, realizamos uma verificação simples para confirmar se os diretórios de saída foram criados corretamente.
Essa etapa ajuda a garantir que a Gold já está pronta para a carga no banco local e para a integração com os módulos de backend e IA.

In [0]:
for table_name in tables_to_export:
    export_name = table_name.split(".")[-1]
    output_path = f"{EXPORT_BASE_PATH}/{export_name}"
    print(f"\nArquivos gerados para {export_name}:")
    display(dbutils.fs.ls(output_path))


Arquivos gerados para dim_cliente:


path,name,size,modificationTime
dbfs:/Volumes/stack_overgol/default/gold_exports/dim_cliente/_SUCCESS,_SUCCESS,0,1778092573000
dbfs:/Volumes/stack_overgol/default/gold_exports/dim_cliente/_committed_7055984973837848287,_committed_7055984973837848287,113,1778092573000
dbfs:/Volumes/stack_overgol/default/gold_exports/dim_cliente/_started_7055984973837848287,_started_7055984973837848287,0,1778092572000
dbfs:/Volumes/stack_overgol/default/gold_exports/dim_cliente/part-00000-tid-7055984973837848287-8576d798-3ef6-4098-a38f-04ddfd37d569-401-1-c000.csv,part-00000-tid-7055984973837848287-8576d798-3ef6-4098-a38f-04ddfd37d569-401-1-c000.csv,11797956,1778092573000



Arquivos gerados para dim_produto:


path,name,size,modificationTime
dbfs:/Volumes/stack_overgol/default/gold_exports/dim_produto/_SUCCESS,_SUCCESS,0,1778092575000
dbfs:/Volumes/stack_overgol/default/gold_exports/dim_produto/_committed_6197756929721414013,_committed_6197756929721414013,113,1778092575000
dbfs:/Volumes/stack_overgol/default/gold_exports/dim_produto/_started_6197756929721414013,_started_6197756929721414013,0,1778092575000
dbfs:/Volumes/stack_overgol/default/gold_exports/dim_produto/part-00000-tid-6197756929721414013-04dc1686-19bc-43fc-a80d-8a6cf7b5bd0b-402-1-c000.csv,part-00000-tid-6197756929721414013-04dc1686-19bc-43fc-a80d-8a6cf7b5bd0b-402-1-c000.csv,58726,1778092575000



Arquivos gerados para dm_cliente_360:


path,name,size,modificationTime
dbfs:/Volumes/stack_overgol/default/gold_exports/dm_cliente_360/_SUCCESS,_SUCCESS,0,1778092577000
dbfs:/Volumes/stack_overgol/default/gold_exports/dm_cliente_360/_committed_1198955506740215381,_committed_1198955506740215381,113,1778092577000
dbfs:/Volumes/stack_overgol/default/gold_exports/dm_cliente_360/_started_1198955506740215381,_started_1198955506740215381,0,1778092576000
dbfs:/Volumes/stack_overgol/default/gold_exports/dm_cliente_360/part-00000-tid-1198955506740215381-0006e1b4-6467-4e9a-8d1d-c3243c3303cf-403-1-c000.csv,part-00000-tid-1198955506740215381-0006e1b4-6467-4e9a-8d1d-c3243c3303cf-403-1-c000.csv,19271330,1778092576000



Arquivos gerados para dm_produto_360:


path,name,size,modificationTime
dbfs:/Volumes/stack_overgol/default/gold_exports/dm_produto_360/_SUCCESS,_SUCCESS,0,1778092579000
dbfs:/Volumes/stack_overgol/default/gold_exports/dm_produto_360/_committed_4035966603736451315,_committed_4035966603736451315,113,1778092579000
dbfs:/Volumes/stack_overgol/default/gold_exports/dm_produto_360/_started_4035966603736451315,_started_4035966603736451315,0,1778092578000
dbfs:/Volumes/stack_overgol/default/gold_exports/dm_produto_360/part-00000-tid-4035966603736451315-c17fc197-4f70-4033-b4fc-7bc0a4bedd57-404-1-c000.csv,part-00000-tid-4035966603736451315-c17fc197-4f70-4033-b4fc-7bc0a4bedd57-404-1-c000.csv,169505,1778092578000



Arquivos gerados para dm_vendas_periodo:


path,name,size,modificationTime
dbfs:/Volumes/stack_overgol/default/gold_exports/dm_vendas_periodo/_SUCCESS,_SUCCESS,0,1778092581000
dbfs:/Volumes/stack_overgol/default/gold_exports/dm_vendas_periodo/_committed_3648476812954350445,_committed_3648476812954350445,113,1778092581000
dbfs:/Volumes/stack_overgol/default/gold_exports/dm_vendas_periodo/_started_3648476812954350445,_started_3648476812954350445,0,1778092579000
dbfs:/Volumes/stack_overgol/default/gold_exports/dm_vendas_periodo/part-00000-tid-3648476812954350445-0ead472e-4975-4ece-aec8-e26b659c43b2-405-1-c000.csv,part-00000-tid-3648476812954350445-0ead472e-4975-4ece-aec8-e26b659c43b2-405-1-c000.csv,37681573,1778092580000


###Otimização física das tabelas Gold
As tabelas Gold serão utilizadas com frequência por dashboards, páginas do CRM e consultas geradas pelo agente Text-to-SQL. Por isso, aplicamos otimizações físicas nas tabelas de maior valor analítico para melhorar a leitura dos dados e organizar melhor os arquivos Delta.

#### Como foi feito
Usamos:
- `OPTIMIZE`, para compactação de arquivos;
- `ZORDER`, para melhorar a localização de dados em colunas usadas com frequência em filtros e joins.

In [0]:
%sql

OPTIMIZE gold.dim_cliente
ZORDER BY (id_cliente, estado_cliente);

OPTIMIZE gold.dim_produto
ZORDER BY (id_produto, categoria_produto);

OPTIMIZE gold.dm_cliente_360
ZORDER BY (id_cliente, estado_cliente);

OPTIMIZE gold.dm_produto_360
ZORDER BY (id_produto, categoria_produto);

OPTIMIZE gold.dm_vendas_periodo
ZORDER BY (data_pedido, estado_cliente, categoria_produto, status_pedido);

path,metrics
,"List(0, 0, List(null, null, 0.0, 0, 0), List(null, null, 0.0, 0, 0), 0, List(minCubeSize(107374182400), List(0, 0), List(1, 4321130), 0, List(0, 0), 0, null), null, 0, 0, 1, 1, false, 0, 0, 1778092924363, 1778092924905, 8, 0, null, List(0, 0), null, 18, 18, 0, 0, null, null)"


### validação final da camada Gold
Antes de considerar a camada Gold pronta para consumo, executamos algumas verificações de sanidade para validar:

- existência das tabelas;
- unicidade do grão esperado;
- volume de registros;
- consistência básica das métricas.

Esses testes são leves, mas ajudam a reduzir o risco de erros estruturais no CRM e no agente de IA.

In [0]:
%sql
SHOW TABLES IN gold;

database,tableName,isTemporary
gold,dim_cliente,false
gold,dim_produto,false
gold,dm_cliente_360,false
gold,dm_produto_360,false
gold,dm_vendas_periodo,false


In [0]:
%sql
SELECT
    COUNT(*) AS total_linhas,
    COUNT(DISTINCT id_cliente) AS clientes_unicos
FROM gold.dim_cliente;

total_linhas,clientes_unicos
58322,58322


In [0]:
%sql
SELECT
    COUNT(*) AS total_linhas,
    COUNT(DISTINCT id_produto) AS produtos_unicos
FROM gold.dim_produto;

total_linhas,produtos_unicos
517,517


In [0]:
%sql
SELECT
    COUNT(*) AS total_linhas,
    COUNT(DISTINCT id_cliente) AS clientes_unicos
FROM gold.dm_cliente_360;

total_linhas,clientes_unicos
58322,58322


In [0]:
%sql
SELECT
    COUNT(*) AS total_linhas,
    COUNT(DISTINCT id_produto) AS produtos_unicos
FROM gold.dm_produto_360;

total_linhas,produtos_unicos
517,517


In [0]:
%sql
SELECT
    MIN(data_pedido) AS menor_data,
    MAX(data_pedido) AS maior_data,
    COUNT(*) AS total_linhas
FROM gold.dm_vendas_periodo;

menor_data,maior_data,total_linhas
2023-01-01,2026-05-31,301061


In [0]:
%sql
SELECT *
FROM gold.dm_vendas_periodo
ORDER BY data_pedido DESC
LIMIT 20;

data_pedido,ano_pedido,mes_pedido,mes_referencia,estado_cliente,cidade_cliente,categoria_produto,status_pedido,metodo_pagamento,total_pedidos,total_clientes,total_produtos,total_itens,receita_total,ticket_medio,receita_entregue,receita_cancelada,receita_reembolsada
2026-05-31,2026,5,2026-05-01T00:00:00.000Z,MINAS GERAIS,Belo Horizonte,Beleza,Aprovado,Pix,1,1,1,3,63.53,63.53,0.0,0.0,0.0
2026-05-31,2026,5,2026-05-01T00:00:00.000Z,MINAS GERAIS,Governador Valadares,Outros,Aprovado,Pix,1,1,1,2,2947.29,2947.29,0.0,0.0,0.0
2026-05-31,2026,5,2026-05-01T00:00:00.000Z,MINAS GERAIS,Itabirito,Esportes,Aprovado,Pix,1,1,1,1,39.05,39.05,0.0,0.0,0.0
2026-05-31,2026,5,2026-05-01T00:00:00.000Z,PARANÁ,Floresta,Outros,Aprovado,Pix,1,1,1,4,3838.46,3838.46,0.0,0.0,0.0
2026-05-31,2026,5,2026-05-01T00:00:00.000Z,SANTA CATARINA,Schroeder,Vestuário,Reembolsado,Cartão,1,1,1,1,129.92,129.92,0.0,0.0,129.92
2026-05-31,2026,5,2026-05-01T00:00:00.000Z,SÃO PAULO,São Paulo,Beleza,Aprovado,Cartão,1,1,1,3,535.19,535.19,0.0,0.0,0.0
2026-05-31,2026,5,2026-05-01T00:00:00.000Z,PARANÁ,Agudos do Sul,Eletrônicos,Aprovado,Pix,1,1,1,5,0.0,null,0.0,0.0,0.0
2026-05-31,2026,5,2026-05-01T00:00:00.000Z,PARAÍBA,São João do Cariri,Eletrônicos,Aprovado,Pix,1,1,1,5,14046.75,14046.75,0.0,0.0,0.0
2026-05-31,2026,5,2026-05-01T00:00:00.000Z,SÃO PAULO,Mauá,Casa,Aprovado,Boleto,1,1,1,1,15.65,15.65,0.0,0.0,0.0
2026-05-31,2026,5,2026-05-01T00:00:00.000Z,RIO DE JANEIRO,Rio de Janeiro,Eletrônicos,Aprovado,Boleto,1,1,1,2,143.21,143.21,0.0,0.0,0.0


#### Tabelas finais da Gold
As principais entregas desta camada são:

- `gold.dim_cliente`
- `gold.dim_produto`
- `gold.dm_cliente_360`
- `gold.dm_produto_360`
- `gold.dm_vendas_periodo`